# Installing Dependencies


In [ ]:
print("Setting up the environment ...")

print("Installing graphviz ...")

#Note: if you are running this code on a local machine, please make sure to run the following command (without the 1>/dev/null) in your terminal to install graphviz before running this code:
!sudo apt-get install graphviz 1>/dev/null 

print("Downloading MCUNet codebase ...")
!wget https://www.dropbox.com/s/3y2n2u3mfxczwcb/mcunetv2-dev-main.zip?dl=0 >/dev/null
!unzip mcunetv2-dev-main.zip* 1>/dev/null
!mv mcunetv2-dev-main/* . 1>/dev/null

!pip install Pillow tqdm numpy torch torchvision matplotlib graphviz 1>/dev/null

In [ ]:
import argparse
import json
from PIL import Image
from tqdm import tqdm
import copy
import math
import numpy as np
import os
import random
import torch
from torch import nn
from torchvision import datasets, transforms
from mcunet.tinynas.search.accuracy_predictor import (
    AccuracyDataset,
    MCUNetArchEncoder,
)

from mcunet.tinynas.elastic_nn.networks.ofa_mcunets import OFAMCUNets
from mcunet.utils.mcunet_eval_helper import calib_bn, validate
from mcunet.utils.arch_visualization_helper import draw_arch


%matplotlib inline
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Data Loader

In [ ]:
def build_val_data_loader(data_dir, resolution, batch_size=128, split=0):
    # split = 0: real val set, split = 1: holdout validation set
    assert split in [0, 1]
    normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    kwargs = {"num_workers": min(8, os.cpu_count()), "pin_memory": False}

    val_transform = transforms.Compose(
        [
            transforms.Resize(
                (resolution, resolution)
            ),  # if center crop, the person might be excluded
            transforms.ToTensor(),
            normalize,
        ]
    )
    val_dataset = datasets.ImageFolder(data_dir, transform=val_transform)

    val_dataset = torch.utils.data.Subset(
        val_dataset, list(range(len(val_dataset)))[split::2]
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, **kwargs
    )
    return val_loader

In [ ]:
data_dir = "VWW/vww-s256/val"

val_data_loader = build_val_data_loader(data_dir, resolution=128, batch_size=1)

vis_x, vis_y = 3, 3
fig, axs = plt.subplots(vis_x, vis_y)

num_images = 0
for data, label in val_data_loader:
    img = np.array((((data + 1) / 2) * 255).numpy(), dtype=np.uint8)
    img = img[0].transpose(1, 2, 0)
    if label.item() == 0:
        label_text = "No person"
    else:
        label_text = "Person"
    axs[num_images // vis_y][num_images % vis_y].imshow(img)
    axs[num_images // vis_y][num_images % vis_y].set_title(f"Label: {label_text}")
    axs[num_images // vis_y][num_images % vis_y].set_xticks([])
    axs[num_images // vis_y][num_images % vis_y].set_yticks([])
    num_images += 1
    if num_images > vis_x * vis_y - 1:
        break

plt.show()

# Super Network OFA

In [ ]:
device = "cuda:0"

ofa_network = OFAMCUNets(
    n_classes=2,
    bn_param=(0.1, 1e-3),
    dropout_rate=0.0,
    base_stage_width="mcunet384",
    width_mult_list=[0.5, 0.75, 1.0],
    ks_list=[3, 5, 7],
    expand_ratio_list=[3, 4, 6],
    depth_list=[0, 1, 2],
    base_depth=[1, 2, 2, 2, 2],
    fuse_blk1=True,
    se_stages=[False, [False, True, True, True], True, True, True, False],
)

ofa_network.load_state_dict(
    torch.load("vww_supernet.pth", map_location="cpu")["state_dict"], strict=True
)

ofa_network = ofa_network.to(device)

# Helper Functions


In [ ]:
from mcunet.utils.pytorch_utils import count_peak_activation_size, count_net_flops, count_parameters

def evaluate_sub_network(ofa_network, cfg, image_size=None):
    if "image_size" in cfg:
        image_size = cfg["image_size"]
    batch_size = 128

    ofa_network.set_active_subnet(**cfg)
    subnet = ofa_network.get_active_subnet().to(device)

    peak_memory = count_peak_activation_size(subnet, (1, 3, image_size, image_size))
    macs = count_net_flops(subnet, (1, 3, image_size, image_size))
    params = count_parameters(subnet)

    calib_bn(subnet, data_dir, batch_size, image_size)

    val_loader = build_val_data_loader(data_dir, image_size, batch_size)
    acc = validate(subnet, val_loader)
    
    return acc, peak_memory, macs, params

In [ ]:
def visualize_subnet(cfg):
    draw_arch(cfg["ks"], cfg["e"], cfg["d"], cfg["image_size"], out_name="viz/subnet")
    im = Image.open("viz/subnet.png")
    im = im.rotate(90, expand=1)
    fig = plt.figure(figsize=(im.size[0] / 250, im.size[1] / 250))
    plt.axis("off")
    plt.imshow(im)
    plt.show()

# **Predictors**

## Efficiency

In [ ]:
class AnalyticalEfficiencyPredictor:
    def __init__(self, net):
        self.net = net

    def get_efficiency(self, spec: dict):
        self.net.set_active_subnet(**spec)
        subnet = self.net.get_active_subnet()
        if torch.cuda.is_available():
            subnet = subnet.cuda()

        image_size = spec.get("image_size")
        data_shape = (1, 3, image_size, image_size)
        macs = count_net_flops(subnet, data_shape) #in millions (M)
        peak_memory = count_peak_activation_size(subnet, data_shape) #in KB

        return dict(millionMACs=macs / 1e6, KBPeakMemory=peak_memory / 1024)

    def satisfy_constraint(self, measured: dict, target: dict):
        for key in measured:
            if key not in target:
                continue
            if measured[key] > target[key]:
                return False
        return True

In [ ]:
efficiency_predictor = AnalyticalEfficiencyPredictor(ofa_network)

macs_smallest_subnet = 8302128.0 / 1000000.0
peak_memory_smallest_subnet = 73728.0 / 1000.0

macs_largest_subnet = 79416432.0 / 1000000.0
peak_memory_largest_subnet = 276480.0 / 1000.0

image_size = 96

smallest_cfg = ofa_network.sample_active_subnet(sample_function=min, image_size=image_size)
eff_smallest = efficiency_predictor.get_efficiency(smallest_cfg)

largest_cfg = ofa_network.sample_active_subnet(sample_function=max, image_size=image_size)
eff_largest = efficiency_predictor.get_efficiency(largest_cfg)

print("Efficiency stats of the smallest subnet:", eff_smallest)
print("Efficiency stats of the largest subnet:", eff_largest)

## Accuracy

In [ ]:
image_size_list = [96, 112, 128, 144, 160]
arch_encoder = MCUNetArchEncoder(
    image_size_list=image_size_list,
    base_depth=ofa_network.base_depth,
    depth_list=ofa_network.depth_list,
    expand_list=ofa_network.expand_ratio_list,
    width_mult_list=ofa_network.width_mult_list,
)

In [ ]:
class AccuracyPredictor(nn.Module):
    def __init__(
        self,
        arch_encoder,
        hidden_size=400,
        n_layers=3,
        checkpoint_path=None,
        device="cuda:0",
    ):
        super(AccuracyPredictor, self).__init__()
        self.arch_encoder = arch_encoder
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.device = device

        layers = []

        for i in range(self.n_layers):
            if i == 0:
                layers.append(nn.Linear(self.arch_encoder.n_dim, self.hidden_size))
            else:
                layers.append(nn.Linear(self.hidden_size, self.hidden_size))

        layers.append(nn.Linear(self.hidden_size, 1, bias=False))
        self.layers = nn.Sequential(*layers)
        self.base_acc = nn.Parameter(
            torch.zeros(1, device=self.device), requires_grad=False
        )

        if checkpoint_path is not None and os.path.exists(checkpoint_path):
            checkpoint = torch.load(checkpoint_path, map_location="cpu")
            if "state_dict" in checkpoint:
                checkpoint = checkpoint["state_dict"]
            self.load_state_dict(checkpoint)
            print("Loaded checkpoint from %s" % checkpoint_path)

        self.layers = self.layers.to(self.device)

    def forward(self, x):
        y = self.layers(x).squeeze()
        return y + self.base_acc

    def predict_acc(self, arch_dict_list):
        X = [self.arch_encoder.arch2feature(arch_dict) for arch_dict in arch_dict_list]
        X = torch.tensor(np.array(X)).float().to(self.device)
        return self.forward(X)

In [ ]:
os.makedirs("pretrained", exist_ok=True)
acc_pred_checkpoint_path = (
    f"pretrained/{ofa_network.__class__.__name__}_acc_predictor.pth"
)
acc_predictor = AccuracyPredictor(
    arch_encoder,
    hidden_size=400,
    n_layers=3,
    checkpoint_path=None,
    device=device,
)
print(acc_predictor)

### Example

In [ ]:
acc_dataset = AccuracyDataset("acc_datasets")
train_loader, valid_loader, base_acc = acc_dataset.build_acc_data_loader(
    arch_encoder=arch_encoder
)

print(f"The basic accuracy (mean accuracy of all subnets within the dataset is: {(base_acc * 100): .1f}%.")

# Let's print one sample in the training set
sampled = 0
for (data, label) in train_loader:
    data = data.to(device)
    label = label.to(device)

    print("=" * 100)

    arch_encoding = arch_encoder.feature2arch(data[0].int().cpu().numpy(), verbose=False)
    arch_encoding = arch_encoder.feature2arch(data[0].int().cpu().numpy(), verbose=True)
    
    visualize_subnet(arch_encoding)
    
    print(f"The accuracy of this subnet on the holdout validation set is: {(label[0] * 100): .1f}%.")
    
    sampled += 1
    if sampled == 1:
        break


In [ ]:
criterion = torch.nn.L1Loss().to(device)
optimizer = torch.optim.Adam(acc_predictor.parameters())

acc_predictor.base_acc.data += base_acc
for epoch in tqdm(range(10)):
    acc_predictor.train()
    for (data, label) in tqdm(train_loader, desc="Epoch%d" % (epoch + 1), position=0, leave=True):
        data = data.to(device)
        label = label.to(device)

        pred = acc_predictor(data)
        loss = criterion(pred, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc_predictor.eval()
    with torch.no_grad():
        with tqdm(total=len(valid_loader), desc="Val", position=0, leave=True) as t:
            for (data, label) in valid_loader:
                data = data.to(device)
                label = label.to(device)

                pred = acc_predictor(data)
                loss = criterion(pred, label)

                t.set_postfix({"loss": loss.item()})
                t.update(1)

if not os.path.exists(acc_pred_checkpoint_path):
    torch.save(acc_predictor.cpu().state_dict(), acc_pred_checkpoint_path)

### Predictor Evaluation

In [ ]:
predicted_accuracies = []
ground_truth_accuracies = []
acc_predictor = acc_predictor.to("cuda:0")
acc_predictor.eval()
with torch.no_grad():
    with tqdm(total=len(valid_loader), desc="Val") as t:
        for (data, label) in valid_loader:
            data = data.to(device)
            label = label.to(device)
            pred = acc_predictor(data)
            predicted_accuracies += pred.cpu().numpy().tolist()
            ground_truth_accuracies += label.cpu().numpy().tolist()
            if len(predicted_accuracies) > 200:
                break
plt.scatter(predicted_accuracies, ground_truth_accuracies)

min_acc, max_acc = min(predicted_accuracies), max(predicted_accuracies)
plt.plot([min_acc, max_acc], [min_acc, max_acc], c="red", linewidth=2)
plt.xlabel("Predicted accuracy")
plt.ylabel("Measured accuracy")
plt.title("Correlation between predicted accuracy and real accuracy")

# Evolution Searcher

# Helper Functions from Robust Analysis


In [ ]:
def get_cfg(agent, constraint, **kwargs):
    best_info = agent.run_search(constraint, **kwargs)
    cfg = best_info[1]

    ofa_network.set_active_subnet(**cfg)
    subnet = ofa_network.get_active_subnet().to(device)
    calib_bn(subnet, data_dir, 128, cfg["image_size"])
    val_loader = build_val_data_loader(data_dir, cfg["image_size"], 128)
    acc = validate(subnet, val_loader)
    visualize_subnet(cfg)
    print(f"Accuracy of the selected subnet: {acc}")

    return acc, subnet, cfg

In [ ]:
def eval_corruptions(cfgs):
    corruptions = [
        "Brightness", "Gaussian Noise", "Contrast", "Defocus Blur", "Elastic", "Fog",
        "Frost", "Glass Blur", "Impulse Noise", "JPEG", "Motion Blur", "Pixelate",
        "Shot Noise", "Snow", "Zoom Blur"
    ]
    base_path = "VWW-C/"
    clean_dir = "VWW/vww-s256/val"
    results = {}
    corr_accs = []

    for i, cfg in enumerate(cfgs):
        ofa_network.set_active_subnet(**cfg)
        subnet     = ofa_network.get_active_subnet().to(device)
        image_size = cfg["image_size"]
        calib_bn(subnet, clean_dir, 128, image_size)

        for corr in corruptions:
            val_loader = build_val_data_loader(f"{base_path}/{corr}/5", image_size, 128)
            acc = validate(subnet, val_loader)
            corr_accs.append(acc)
        
        mean_acc_corr = sum(corr_accs) / len(corr_accs)
        results[i] = mean_acc_corr


    return results  # {0: mca_0, 1: mca_1, ...}

In [ ]:
class EvolutionSearcher:
    def __init__(self, efficiency_predictor, accuracy_predictor, **kwargs):
        self.efficiency_predictor = efficiency_predictor
        self.accuracy_predictor = accuracy_predictor

        # evolution hyper-parameters
        self.arch_mutate_prob = kwargs.get("arch_mutate_prob", 0.1)
        self.resolution_mutate_prob = kwargs.get("resolution_mutate_prob", 0.5)
        self.population_size = kwargs.get("population_size", 100)
        self.max_time_budget = kwargs.get("max_time_budget", 500)
        self.parent_ratio = kwargs.get("parent_ratio", 0.25)
        self.mutation_ratio = kwargs.get("mutation_ratio", 0.5)

    def update_hyper_params(self, new_param_dict):
        self.__dict__.update(new_param_dict)

    def random_valid_sample(self, constraint):
        # randomly sample subnets until finding one that satisfies the constraint
        while True:
            sample = self.accuracy_predictor.arch_encoder.random_sample_arch()
            efficiency = self.efficiency_predictor.get_efficiency(sample)
            if self.efficiency_predictor.satisfy_constraint(efficiency, constraint):
                return sample, efficiency

    def mutate_sample(self, sample, constraint):
        while True:
            new_sample = copy.deepcopy(sample)

            self.accuracy_predictor.arch_encoder.mutate_resolution(new_sample, self.resolution_mutate_prob)
            self.accuracy_predictor.arch_encoder.mutate_width(new_sample, self.arch_mutate_prob)
            self.accuracy_predictor.arch_encoder.mutate_arch(new_sample, self.arch_mutate_prob)

            efficiency = self.efficiency_predictor.get_efficiency(new_sample)
            if self.efficiency_predictor.satisfy_constraint(efficiency, constraint):
                return new_sample, efficiency

    def crossover_sample(self, sample1, sample2, constraint):
        while True:
            new_sample = copy.deepcopy(sample1)
            for key in new_sample.keys():
                if not isinstance(new_sample[key], list):
                    new_sample[key] = random.choice([sample1[key],sample2[key]])
                else:
                    for i in range(len(new_sample[key])):
                        new_sample[key][i] = random.choice([sample1[key][i],sample2[key][i]])
                        
            efficiency = self.efficiency_predictor.get_efficiency(new_sample)
            if self.efficiency_predictor.satisfy_constraint(efficiency, constraint):
                return new_sample, efficiency

    def run_search(self, constraint, **kwargs):
        self.update_hyper_params(kwargs)

        mutation_numbers = int(round(self.mutation_ratio * self.population_size))
        parents_size = int(round(self.parent_ratio * self.population_size))

        best_valids = [-100]
        population = []  # (acc, sample) tuples
        child_pool = []
        best_info = None

        for _ in range(self.population_size):
            sample, efficiency = self.random_valid_sample(constraint)
            child_pool.append(sample)

        accs = self.accuracy_predictor.predict_acc(child_pool)
        corr_accs = eval_corruptions(child_pool)
        for i in range(self.population_size):
            # Fitness Robust
            fitness = (accs[i].item() + corr_accs[i])
            population.append((fitness, child_pool[i]))

        # evolving the population
        with tqdm(total=self.max_time_budget) as t:
            for i in range(self.max_time_budget):
                population = sorted(population, key=lambda x: x[0], reverse=True)
                population = population[:parents_size]

                acc = population[0][0]
                if acc > best_valids[-1]:
                    best_valids.append(acc)
                    best_info = population[0]
                else:
                    best_valids.append(best_valids[-1])

                child_pool = []
                for j in range(mutation_numbers):
                    par_sample = population[np.random.randint(parents_size)][1]
                
                    new_sample, efficiency = self.mutate_sample(par_sample, constraint)
                    child_pool.append(new_sample)

                for j in range(self.population_size - mutation_numbers):
                    par_sample1 = population[np.random.randint(parents_size)][1]
                    par_sample2 = population[np.random.randint(parents_size)][1]
                    # crossover
                    new_sample, efficiency = self.crossover_sample(
                        par_sample1, par_sample2, constraint
                    )
                    child_pool.append(new_sample)
                # predict accuracy with the accuracy predictor
                accs = self.accuracy_predictor.predict_acc(child_pool)
                corr_accs = eval_corruptions(child_pool)
                for j in range(self.population_size):
                    fitness = (accs[j].item() + corr_accs[j])
                    population.append((fitness, child_pool[j]))

                t.update(1)

        return best_info

In [ ]:
evo_params = {
    'arch_mutate_prob': 0.1,
    'resolution_mutate_prob': 0.3,
    'population_size': 20,
    'max_time_budget': 10,
    'parent_ratio': 0.2,
    'mutation_ratio': 0.2,
}

nas_agent = EvolutionSearcher(efficiency_predictor, acc_predictor, **evo_params)

### Search of the best subnets with 52M MACs and 128KB of RAM

We will evaluate 10 samples, where each sample represents a sub-network obtained through the evolutionary search using the new fitness function, which combines clean dataset accuracy with the mean corruption accuracy

In [ ]:
subnets_mac_mem = {}

In [ ]:
cfg_52_128_acc_list = []
cfg_52_128_subnet_list = []
cfg_52_128_cfg_list = []

for _ in range (10):
    (millionMACs, KBPeakMemory) = [52, 128]
    print(f"Evolution search with constraint: MACs <= {millionMACs}M, peak memory <= {KBPeakMemory}KB")
    subnets_mac_mem["52_128"] = get_cfg(nas_agent, dict(millionMACs=millionMACs, KBPeakMemory=KBPeakMemory))
    print("Evolution search finished!")
    cfg_52_128_acc_list.append(subnets_mac_mem["52_128"][0])
    cfg_52_128_subnet_list.append(subnets_mac_mem["52_128"][1])
    cfg_52_128_cfg_list.append(subnets_mac_mem["52_128"][2])

    with open("cfg_52_128_acc_ROBUST.json", "w") as f:
        json.dump(cfg_52_128_acc_list, f, indent=2)

    with open("cfg_52_128_cfg_ROBUST.json", "w") as f:    
        json.dump(cfg_52_128_cfg_list, f, indent=2)

# Recalculating Accuracy and MCA

In [ ]:
def test_clean(cfgs, filename):
    accs = []
    data_dir = "VWW/vww-s256/val"
    for cfg in cfgs:
        ofa_network.set_active_subnet(**cfg)
        subnet = ofa_network.get_active_subnet().to(device)
        calib_bn(subnet, data_dir, 128, cfg["image_size"])
        val_loader = build_val_data_loader(data_dir, cfg["image_size"], 128)
        acc = validate(subnet, val_loader)
        accs.append(acc)
    with open(filename, "w") as f:
        json.dump(accs, f, indent=2)

In [ ]:
def test_corruptions(cfgs, filename):
    corruptions = [
        "Brightness", "Contrast", "Defocus Blur", "Elastic", "Fog",
        "Frost", "Gaussian Noise", "Glass Blur", "Impulse Noise", "JPEG", "Motion Blur", "Pixelate",
        "Shot Noise", "Snow", "Zoom Blur"
    ]
    levels = [5]
    base_path = "VWW-C/"
    results = {}

    for i, cfg in enumerate(cfgs):
        cfg_key = f"sample_{i}"
        results[cfg_key] = {"cfg": cfg, "corruptions": {}}

        ofa_network.set_active_subnet(**cfg)
        subnet = ofa_network.get_active_subnet().to(device)
        image_size = cfg["image_size"]
        calib_bn(subnet, "VWW/vww-s256/val", 128, image_size)

        for corruption in corruptions:
            results[cfg_key]["corruptions"][corruption] = {}

            for level in levels:
                data_dir = f"{base_path}/{corruption}/{level}"
                val_loader = build_val_data_loader(data_dir, image_size, 128)
                acc = validate(subnet, val_loader)
                results[cfg_key]["corruptions"][corruption][level] = acc
                print(f"sample_{i} | {corruption} | level {level} → acc={acc:.2f}%")

    mean = sum(results[cfg_key]["corruptions"][corr][5] for corr in corruptions) / len(corruptions)
    print(f"Mean accuracy across all corruptions: {mean:.2f}%")
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)


In [ ]:
test_clean(cfg_52_128_cfg_list, "ACCS_52_128_clean_ROBUST.json")
test_corruptions(cfg_52_128_cfg_list, "ACCS_52_128_corrupted_ROBUST.json")